# End-to-End LangGraph Support System Test

This notebook tests VADER sentiment, Chroma ingestion/retrieval, the four routing actions, groundedness refinement, HITL, memory, and audit logging.

In [1]:
from pathlib import Path
import os, sys, json

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print('Project root:', ROOT)


Project root: C:\Users\Salome Purra\PycharmProjects\support-ticket-agent\src


## 1. Local components: VADER + vector-store ingestion/retrieval
This cell does not call the OpenAI API.

In [2]:
from tools.sentiment_tool import sentiment_classification_tool
from rag.ingest import ingest_knowledge_base
from rag.retriever import retrieve_documents

print(sentiment_classification_tool.invoke({'text': 'I am very frustrated with this issue.'}))
print(ingest_knowledge_base())
docs = retrieve_documents('expired password reset link', k=3)
[(d['source'], d['chunk_id']) for d in docs]


C:\Users\Salome Purra\PycharmProjects\support-ticket-agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'label': 'negative', 'compound': -0.5709, 'scores': {'neg': 0.381, 'neu': 0.619, 'pos': 0.0, 'compound': -0.5709}, 'model': 'VADER'}
{'total_chunks': 5, 'new_chunks_added': 0}


[('account_access_faq.md', 1),
 ('abusive_content_policy.md', 0),
 ('troubleshooting_faq.md', 4)]

## 2. Check environment
Create `.env` in the project root before running the full graph.

In [3]:
from dotenv import load_dotenv
load_dotenv(ROOT / '.env')
assert os.getenv('OPENAI_API_KEY'), 'OPENAI_API_KEY is missing from .env'
assert os.getenv('OPENAI_MODEL'), 'OPENAI_MODEL is missing from .env'
print('Model:', os.getenv('OPENAI_MODEL'))


Model: gpt-4.1-mini


## 3. Load synthetic ticket queue

In [9]:
print(ROOT.parent )
tickets = json.loads((ROOT.parent / "data" / "tickets" / "synthetic_tickets.json")
    .read_text(encoding="utf-8"))
[(t['ticket_id'], t['subject']) for t in tickets]

C:\Users\Salome Purra\PycharmProjects\support-ticket-agent


[('TKT-1001', 'Duplicate billing charge'),
 ('TKT-1002', 'Expired password reset link'),
 ('TKT-1003', 'How do I exploit refunds?'),
 ('TKT-1004', 'Write a threatening message'),
 ('TKT-1005', 'App crashes'),
 ('TKT-1006', 'Unknown loyalty policy')]

## 4. Run one ticket to the HITL interrupt

In [10]:
from graph import graph

ticket = tickets[1]  # expired reset link
config = {'configurable': {'thread_id': ticket['customer_id']}}
result = graph.invoke({'ticket': ticket}, config=config)
print('Interrupted:', bool(result.get('__interrupt__')))
if result.get('__interrupt__'):
    review = result['__interrupt__'][0].value
    print(json.dumps(review, indent=2, default=str))


2026-08-21 15:56:20,610 - httpx2 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-08-21 15:56:20,628 - openai._base_client - INFO - Retrying request to /chat/completions in 0.441808 seconds
2026-08-21 15:56:21,415 - httpx2 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-08-21 15:56:21,416 - openai._base_client - INFO - Retrying request to /chat/completions in 0.808937 seconds
2026-08-21 15:56:23,259 - httpx2 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"


RateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}

## 5. Approve the draft and resume the same thread
This records approval and audit state only; it does **not** send a customer message.

In [ ]:
from langgraph.types import Command

if result.get('__interrupt__'):
    result = graph.invoke(
        Command(resume={'decision': 'approve'}),
        config=config,
    )

print({
    'route': result.get('route'),
    'status': result.get('final_status'),
    'audit_event_id': result.get('audit_event_id'),
})


## 6. Exercise the full synthetic queue
For notebook testing, this simulates a human reviewer approving each presented draft. The system still pauses at `interrupt()` before each approval.

In [ ]:
summary = []

for ticket in tickets:
    cfg = {'configurable': {'thread_id': ticket['customer_id']}}
    out = graph.invoke({'ticket': ticket}, config=cfg)

    while out.get('__interrupt__'):
        # Simulated reviewer action for test purposes only.
        out = graph.invoke(
            Command(resume={'decision': 'approve'}),
            config=cfg,
        )

    summary.append({
        'ticket_id': ticket['ticket_id'],
        'subject': ticket['subject'],
        'sentiment': out.get('sentiment', {}).get('label'),
        'route': out.get('route'),
        'groundedness': out.get('groundedness_score'),
        'status': out.get('final_status'),
    })

summary


## 7. Inspect audit events

In [ ]:
import sqlite3
from audit import AUDIT_DB

with sqlite3.connect(AUDIT_DB) as conn:
    rows = conn.execute(
        'SELECT event_id, ticket_id, route, human_decision, final_status, created_at '
        'FROM audit_events ORDER BY created_at DESC LIMIT 20'
    ).fetchall()

rows


## 8. Inspect conversation-thread state
The checkpointer preserves state for the stable `thread_id` while this Python process is alive.

In [ ]:
snapshot = graph.get_state(config)
print('Next:', snapshot.next)
print('Messages:', len(snapshot.values.get('messages', [])))
[m.content for m in snapshot.values.get('messages', [])[-3:]]
